# Silver - ecommerce_clientes

Desenvolvido por: Ygor Moraes

Este notebook lê a Bronze `ecommerce_clientes` e grava a Silver tratada em Delta.

Regras aplicadas:
- deduplicar clientes por `id_cliente`;
- padronizar `email` com letras minúsculas e sem espaços;
- converter `dt_cadastro` e `dt_ultima_atualizacao` para timestamp;
- remover `senha_hash`, mantendo esse dado sensível apenas na Bronze;
- criar `dias_desde_cadastro`;
- manter particionamento por `ano` e `mes`.

In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
# Define imports, caminhos e parâmetros da Silver de clientes.

from pyspark.sql.functions import (
    col,
    count,
    when,
    trim,
    lower,
    to_timestamp,
    current_timestamp,
    current_date,
    datediff,
    to_date,
    row_number
)

from pyspark.sql.window import Window

BRONZE_TABLE = "ecommerce_clientes"
BRONZE_PATH = f"{BRONZE_BASE_PATH}{BRONZE_TABLE}"

SILVER_TABLE = "ecommerce_clientes"
SILVER_PATH = f"{SILVER_BASE_PATH}{SILVER_TABLE}"

KEY_COLUMNS = ["id_cliente"]

SILVER_WRITE_MODE = "overwrite"

BRONZE_REQUIRED_COLUMNS = [
    "id_cliente",
    "uuid_cliente",
    "nome",
    "sobrenome",
    "email",
    "senha_hash",
    "dt_cadastro",
    "dt_ultima_atualizacao",
    "bronze_ingested_at",
    "bronze_source_file",
    "ano",
    "mes"
]

SILVER_REQUIRED_COLUMNS = [
    "id_cliente",
    "uuid_cliente",
    "nome",
    "sobrenome",
    "email",
    "dt_cadastro",
    "dt_ultima_atualizacao",
    "dias_desde_cadastro",
    "bronze_ingested_at",
    "bronze_source_file",
    "silver_processed_at",
    "ano",
    "mes"
]

adls_options = get_adls_options()

print("Notebook configurado.")
print(f"Origem Bronze: {BRONZE_PATH}")
print(f"Destino Silver: {SILVER_PATH}")
print(f"Modo de escrita: {SILVER_WRITE_MODE}")

In [0]:
# Lê a Bronze e valida se as colunas necessárias existem.

df_bronze = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(BRONZE_PATH)
)

bronze_columns = df_bronze.columns

missing_bronze_columns = [
    c for c in BRONZE_REQUIRED_COLUMNS
    if c not in bronze_columns
]

if missing_bronze_columns:
    raise Exception(f"Colunas obrigatórias ausentes na Bronze: {missing_bronze_columns}")

total_bronze = df_bronze.count()

print("Bronze lida com sucesso.")
print(f"Total de registros na Bronze: {total_bronze}")

df_bronze.printSchema()

In [0]:
# Valida se os campos principais podem ser convertidos antes da transformação.

df_validacao_conversoes = df_bronze.select(
    count("*").alias("total_linhas"),

    count(
        when(
            col("id_cliente").isNotNull()
            & col("id_cliente").cast("int").isNull(),
            True
        )
    ).alias("falhas_id_cliente"),

    count(
        when(
            col("dt_cadastro").isNotNull()
            & to_timestamp(col("dt_cadastro")).isNull(),
            True
        )
    ).alias("falhas_dt_cadastro"),

    count(
        when(
            col("dt_ultima_atualizacao").isNotNull()
            & to_timestamp(col("dt_ultima_atualizacao")).isNull(),
            True
        )
    ).alias("falhas_dt_ultima_atualizacao")
)

display(df_validacao_conversoes)

validacao_conversoes = df_validacao_conversoes.collect()[0]

if validacao_conversoes["falhas_id_cliente"] > 0:
    raise Exception("Existem valores de id_cliente que não podem ser convertidos para integer.")

if validacao_conversoes["falhas_dt_cadastro"] > 0:
    raise Exception("Existem valores de dt_cadastro que não podem ser convertidos para timestamp.")

if validacao_conversoes["falhas_dt_ultima_atualizacao"] > 0:
    raise Exception("Existem valores de dt_ultima_atualizacao que não podem ser convertidos para timestamp.")

print("Validação OK: conversões principais podem ser feitas.")

In [0]:
# Aplica limpeza, tipagem, deduplicação por cliente e remove senha_hash da Silver.

df_silver_base = (
    df_bronze
    .withColumn("id_cliente_int", col("id_cliente").cast("int"))
    .withColumn("dt_cadastro_ts", to_timestamp(col("dt_cadastro")))
    .withColumn("dt_ultima_atualizacao_ts", to_timestamp(col("dt_ultima_atualizacao")))
)

window_deduplicacao = (
    Window
    .partitionBy("id_cliente_int")
    .orderBy(
        col("dt_ultima_atualizacao_ts").desc_nulls_last(),
        col("dt_cadastro_ts").desc_nulls_last(),
        col("bronze_ingested_at").desc_nulls_last()
    )
)

df_silver = (
    df_silver_base
    .withColumn("rn", row_number().over(window_deduplicacao))
    .filter(col("rn") == 1)
    .select(
        col("id_cliente_int").alias("id_cliente"),
        col("uuid_cliente").cast("string").alias("uuid_cliente"),
        trim(col("nome")).alias("nome"),
        trim(col("sobrenome")).alias("sobrenome"),
        lower(trim(col("email"))).alias("email"),
        col("dt_cadastro_ts").alias("dt_cadastro"),
        col("dt_ultima_atualizacao_ts").alias("dt_ultima_atualizacao"),
        datediff(current_date(), to_date(col("dt_cadastro_ts"))).alias("dias_desde_cadastro"),

        col("bronze_ingested_at").cast("timestamp").alias("bronze_ingested_at"),
        col("bronze_source_file").cast("string").alias("bronze_source_file"),

        current_timestamp().alias("silver_processed_at"),

        col("ano").cast("int").alias("ano"),
        col("mes").cast("int").alias("mes")
    )
)

total_silver = df_silver.count()

print("Silver criada em memória.")
print(f"Total de registros na Silver: {total_silver}")

df_silver.printSchema()

In [0]:
# Valida se a Silver possui apenas um registro por id_cliente.

total_ids_distintos_bronze = (
    df_bronze
    .filter(col("id_cliente").isNotNull())
    .select(col("id_cliente").cast("int").alias("id_cliente"))
    .distinct()
    .count()
)

duplicados_silver = (
    df_silver
    .groupBy("id_cliente")
    .count()
    .filter(col("count") > 1)
    .count()
)

print(f"IDs distintos na Bronze: {total_ids_distintos_bronze}")
print(f"Registros na Silver: {total_silver}")
print(f"IDs duplicados na Silver: {duplicados_silver}")

if total_silver != total_ids_distintos_bronze:
    raise Exception("Erro: quantidade da Silver diferente da quantidade de IDs distintos da Bronze.")

if duplicados_silver > 0:
    raise Exception("Erro: ainda existem id_cliente duplicados na Silver.")

print("Validação OK: Silver deduplicada por id_cliente.")

In [0]:
# Valida campos obrigatórios, auditoria e schema antes da gravação.

df_validacao_silver = df_silver.select(
    count("*").alias("total_linhas"),
    count(when(col("id_cliente").isNull(), True)).alias("id_cliente_nulo"),
    count(when(col("dt_cadastro").isNull(), True)).alias("dt_cadastro_nulo"),
    count(when(col("ano").isNull(), True)).alias("ano_nulo"),
    count(when(col("mes").isNull(), True)).alias("mes_nulo"),
    count(when(col("dias_desde_cadastro").isNull(), True)).alias("dias_desde_cadastro_nulo"),
    count(when(col("bronze_ingested_at").isNull(), True)).alias("bronze_ingested_at_nulo"),
    count(when(col("bronze_source_file").isNull(), True)).alias("bronze_source_file_nulo"),
    count(when(col("silver_processed_at").isNull(), True)).alias("silver_processed_at_nulo")
)

display(df_validacao_silver)

validacao_silver = df_validacao_silver.collect()[0]

if validacao_silver["id_cliente_nulo"] > 0:
    raise Exception("Erro: existem registros com id_cliente nulo na Silver.")

if validacao_silver["dt_cadastro_nulo"] > 0:
    raise Exception("Erro: existem registros com dt_cadastro nulo na Silver.")

if validacao_silver["ano_nulo"] > 0:
    raise Exception("Erro: existem registros com ano nulo na Silver.")

if validacao_silver["mes_nulo"] > 0:
    raise Exception("Erro: existem registros com mes nulo na Silver.")

if validacao_silver["dias_desde_cadastro_nulo"] > 0:
    raise Exception("Erro: existem registros sem dias_desde_cadastro.")

if validacao_silver["bronze_ingested_at_nulo"] > 0:
    raise Exception("Erro: existem registros sem bronze_ingested_at.")

if validacao_silver["bronze_source_file_nulo"] > 0:
    raise Exception("Erro: existem registros sem bronze_source_file.")

if validacao_silver["silver_processed_at_nulo"] > 0:
    raise Exception("Erro: existem registros sem silver_processed_at.")

if "senha_hash" in df_silver.columns:
    raise Exception("Erro: senha_hash não deve ser promovida para a Silver.")

print("Validação OK: qualidade mínima da Silver aprovada.")

In [0]:
# Grava a Silver em Delta com overwrite.

(
    df_silver
    .write
    .format("delta")
    .options(**adls_options)
    .option("overwriteSchema", "true")
    .mode(SILVER_WRITE_MODE)
    .partitionBy("ano", "mes")
    .save(SILVER_PATH)
)

print(f"Silver gravada com sucesso em Delta: {SILVER_PATH}")
print(f"Modo de escrita utilizado: {SILVER_WRITE_MODE}")

In [0]:
# Lê a Silver gravada e valida volume, duplicidade e schema final.

df_silver_saved = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(SILVER_PATH)
)

total_silver_saved = df_silver_saved.count()

duplicados_silver_saved = (
    df_silver_saved
    .groupBy("id_cliente")
    .count()
    .filter(col("count") > 1)
    .count()
)

colunas_silver_saved = df_silver_saved.columns

colunas_ausentes = [
    c for c in SILVER_REQUIRED_COLUMNS
    if c not in colunas_silver_saved
]

print(f"IDs distintos na Bronze: {total_ids_distintos_bronze}")
print(f"Total Silver gravada: {total_silver_saved}")
print(f"IDs duplicados na Silver gravada: {duplicados_silver_saved}")

if total_silver_saved != total_ids_distintos_bronze:
    raise Exception("Erro: quantidade da Silver gravada diferente da quantidade de IDs distintos da Bronze.")

if duplicados_silver_saved > 0:
    raise Exception("Erro: existem id_cliente duplicados na Silver gravada.")

if colunas_ausentes:
    raise Exception(f"Erro: colunas obrigatórias ausentes na Silver: {colunas_ausentes}")

if "senha_hash" in colunas_silver_saved:
    raise Exception("Erro: senha_hash não deveria estar na Silver.")

df_silver_saved.printSchema()

print("Validação final da Silver OK.")